<a href="https://colab.research.google.com/github/elijahschoeniger-cpu/Walmart-Project-1-/blob/main/notebooks/2026-10-23%20%E2%80%94%20Midterm%20Checkpoint%20%E2%80%94%20Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DS2002 · Midterm Checkpoint

**Lab — 2026-10-23 · Fall 2026**  

---

## Lab 09 — Midterm checkpoint, week 1 of 3

**What you submit: the link to your team's main repo. Only that.** One submission per team. Do not upload separate files.

Put this notebook in the repo (the `notebooks/` folder is fine) and push it, so everything we check is in one place.

**You do not have to fill this all out.** It is a checkpoint, not a finished project. Skip any section you have not started and write "not started" next to it. What we must see is:

1. **Your charter**, committed to the repo
2. **Any progress** you have made, with code that ran and output you can point at

Start with the team setup, then work down the same eight things we listed at Monday's kickoff, as far as you have gotten. This is graded on evidence, not on intentions.

The purpose is to catch a stalled team while there is still time to unstall it. If something is not working, say so here plainly. A checkpoint that honestly reports "the API join is not working and here is the error" is worth more than one that claims progress that does not exist, and it gets you help on Monday.

### Team setup

This part is required. Fill it in once for the whole team.

You need:

- Your team name, and every member with a GitHub handle (2 to 4 people)
- A link to your team's **main repo**. This is the one link you submit.
- A link to `CHARTER.md` in that repo, the page on GitHub, not a file on someone's laptop

Nice to have, and also in your charter: this week's notebook owner, your meeting time, and the five worst problems in the data from the profile you ran on Monday. Be specific: "`unit_price` is stored as text and some values start with `$`" beats "prices are messy."

In [4]:
team_setup = {
    'team_name': 'Case Crushers',
    'members': [                                  # everyone on the team, 2 to 4 people
        {'name': 'AJ Amos', 'github': 'pizzeman'},
        {'name': 'Sophie Li', 'github': 'Sophieli0113'},
        {'name': 'Eli Schoeniger', 'github': 'elijahschoeniger'},
    ],
    'repo_url': 'https://github.com/elijahschoeniger-cpu/Walmart-Project-1-',                           # your main team repo: https://github.com/...
    'charter_url': 'https://github.com/elijahschoeniger-cpu/Walmart-Project-1-/blob/main/CHARTER.md',                        # link to CHARTER.md in that repo
    'notebook_owner_this_week': 'Sophie Li',           # nice to have
    'meeting_time': 'Monday 2-3pm',                       # nice to have, e.g. 'Tue and Thu 7pm'
    'five_worst_problems': ['timestamp has 886 missing values and mixed formats',
    'store_id has 60 distinct spellings for about 14 real stores',
    'category has 37 distinct values but there are only 10 real categories',
    'unit_price is stored as text and some values start with $',
    '30 exact duplicate rows',],              # nice to have, five specific problems
}

def check_team(t, quiet=False):
    def real(v):
        return isinstance(v, str) and v.strip() != '' and not v.startswith('TODO')
    required = {
        'team name': real(t['team_name']),
        '2 to 4 members, each with a name and GitHub handle':
            2 <= len(t['members']) <= 4 and all(real(m['name']) and real(m['github']) for m in t['members']),
        'main repo link': real(t['repo_url']) and t['repo_url'].startswith('https://github.com/'),
        'charter link points at CHARTER.md': real(t['charter_url']) and t['charter_url'].endswith('CHARTER.md'),
    }
    nice = {
        'notebook owner': real(t['notebook_owner_this_week']),
        'meeting time': real(t['meeting_time']),
        'five specific problems': len(t['five_worst_problems']) >= 5 and all(real(p) for p in t['five_worst_problems']),
    }
    if not quiet:
        print('REQUIRED')
        for label, good in required.items():
            print('  PASS' if good else '  FAIL', '-', label)
        print('NICE TO HAVE')
        for label, good in nice.items():
            print('  done' if good else '  not yet', '-', label)
        print()
        print('team setup complete:', all(required.values()))
    return all(required.values())

check_team(team_setup)

REQUIRED
  PASS - team name
  PASS - 2 to 4 members, each with a name and GitHub handle
  PASS - main repo link
  PASS - charter link points at CHARTER.md
NICE TO HAVE
  done - notebook owner
  done - meeting time
  done - five specific problems

team setup complete: True


True

Now show that the charter is actually committed. This is required. Run the next cell and leave its output in the notebook. Point `REPO_PATH` at your local clone of the team repo. If it prints a line like `a1b2c3d Add team charter`, you are done. If it prints "no commits found," the charter is not in the repo yet, so commit it and push before you submit.

In [5]:
import subprocess

!git clone https://github.com/elijahschoeniger-cpu/Walmart-Project-1-.git

REPO_PATH = '/content/Walmart-Project-1-'      # your local clone of the team repo

result = subprocess.run(['git', '-C', REPO_PATH, 'log', '--oneline', '--', 'CHARTER.md'],
                        capture_output=True, text=True)
print(result.stdout.strip() or 'no commits found for CHARTER.md')

fatal: destination path 'Walmart-Project-1-' already exists and is not an empty directory.
0c9f102 Rename charter.md to CHARTER.md


### 1 — Your cleaning pipeline as functions

**TODO:** paste the cleaning code you have written so far. It should be functions, not loose cells, and it should cover at minimum: duplicates, `unit_price` to numeric (it comes in as text, some with a `$`), the negative `quantity` values with your decision applied, `timestamp` parsed, and `store_id` cleaned up so it will join to the store file.

Print the row count before and after. We want to see how many rows your decisions removed.

In [9]:
import os, subprocess, sqlite3
import numpy as np
import pandas as pd

REPO_URL  = 'https://github.com/elijahschoeniger-cpu/Walmart-Project-1-.git'
REPO_PATH = '/content/Walmart-Project-1-/'
if not os.path.exists(REPO_PATH):
    subprocess.run(['git', 'clone', '-q', REPO_URL, REPO_PATH], check=True)
DATA = os.path.join(REPO_PATH, 'data')
DB_PATH = os.path.join(DATA, 'inventory_and_sales.db')
assert os.path.exists(DB_PATH), f'database not found at {DB_PATH}'

conn = sqlite3.connect(DB_PATH)
print(pd.read_sql_query("SELECT name FROM sqlite_master WHERE type='table'", conn))

DECISIONS = []
def log(step, decision, rows_affected):
    DECISIONS.append({'step': step, 'decision': decision, 'rows': rows_affected})
    print(f'[{step}] {decision} ({rows_affected} row(s))')

def show_log():
    return pd.DataFrame(DECISIONS)

def profile(df, name='dataset'):
    print(f'=== {name} ===')
    print(f'{len(df):,} rows x {len(df.columns)} columns')
    print()
    summary = pd.DataFrame({
        'dtype': df.dtypes.astype(str),
        'nulls': df.isnull().sum(),
        'null_pct': (100 * df.isnull().mean()).round(1),
        'distinct': df.nunique(),
    })
    print(summary)
    print()
    print('exact duplicate rows:', df.duplicated().sum())
    return summary

# Read everything as text so pandas doesn't guess types; we convert explicitly.
NA_TOKENS = ['', 'NULL', 'N/A', 'NaN', 'nan', 'NA', 'None']
raw_txn     = pd.read_csv(os.path.join(DATA, 'walmart_transactions.csv'), dtype=str, keep_default_na=False)
raw_stores  = pd.read_csv(os.path.join(DATA, 'store_locations.csv'), dtype=str, keep_default_na=False)
raw_catalog = pd.read_csv(os.path.join(DATA, 'product_catalog.csv'), dtype=str, keep_default_na=False)
raw_dc      = pd.read_csv(os.path.join(DATA, 'distribution_centers.csv'), dtype=str, keep_default_na=False)
raw_sales   = pd.read_sql_query('SELECT * FROM daily_sales_summary', conn)
raw_inv     = pd.read_sql_query('SELECT * FROM inventory_levels', conn)

# Profile the raw data (default NA handling, so NULL/N/A count as missing)
profile(pd.read_csv(os.path.join(DATA, 'walmart_transactions.csv')), 'walmart_transactions')
profile(pd.read_csv(os.path.join(DATA, 'store_locations.csv')), 'store_locations')


# ===== HELPERS + LOOKUP TABLES =====
def to_na(s):
    """Strip whitespace and turn every missing-value spelling into a real NaN."""
    s = s.astype(str).str.strip()
    return s.mask(s.isin(NA_TOKENS))

def norm_store_id(s):
    """' FL-330', 'fl_330', 'FL330' -> 'FL-330'."""
    return (s.astype(str).str.strip().str.upper()
             .str.replace(r'^FL[\s_-]?(\d{3})$', r'FL-\1', regex=True))

# 45 SKUs collapse to 20 real products
SKU_TO_PRODUCT = {
    'PT-12': 'PT-STRAW', 'SKU#459812': 'PT-STRAW', 'POPTART-STRAW': 'PT-STRAW', 'PT_STRAWBERRY': 'PT-STRAW',
    'WTR-24': 'WATER-24', 'WATER24PK': 'WATER-24', 'WTR_24PK': 'WATER-24',
    'BATT-100': 'BATT-AA4', 'BAT-AA4': 'BATT-AA4', 'BATT_AA_4PK': 'BATT-AA4',
    'FLASH-01': 'FLASHLIGHT', 'LED-FLASH': 'FLASHLIGHT',
    'BEER-24': 'BEER-24', 'LTBEER24': 'BEER-24',
    'BG-MONO': 'MONOPOLY', 'MONOPOLY-01': 'MONOPOLY', 'GAME-MONO': 'MONOPOLY',
    'CARDS-01': 'CARDS', 'PLAYCARD-STD': 'CARDS',
    'STREAM-01': 'STREAMING', 'ROKU-STD': 'STREAMING',
    'SOUP-CAN': 'SOUP', 'CNSOUP-01': 'SOUP',
    'BREAD-WH': 'BREAD', 'WHITEBREAD': 'BREAD',
    'PB-16': 'PB-16', 'PNUTBTR-16': 'PB-16',
    'CNDL-EM': 'CANDLES', 'CANDLE-EMRG': 'CANDLES',
    'FAID-01': 'FIRSTAID', 'FIRSTAID-KIT': 'FIRSTAID',
    'CHRG-PORT': 'CHARGER', 'PORTCHARGER': 'CHARGER',
    'CHIPS-VP': 'CHIPS-VP', 'CHIPVARIETY': 'CHIPS-VP',
    'PTOWEL-6': 'PTOWEL-6', 'PAPERTOWEL6': 'PTOWEL-6',
    'TP-12': 'TP-12', 'TOILETPAPER12': 'TP-12',
    'DIAP-S3': 'DIAPERS-3', 'DIAPERS-3': 'DIAPERS-3',
    'BFORM-01': 'FORMULA', 'BABYFORMULA': 'FORMULA',
    'TARP-HD': 'TARP', 'HDTARP-01': 'TARP',
}
PRODUCTS = pd.DataFrame([
    ('PT-STRAW',  'Pop-Tarts Strawberry',   'Breakfast'),
    ('WATER-24',  'Bottled Water 24-Pack',  'Emergency'),
    ('BATT-AA4',  'AA Batteries 4-Pack',    'Emergency'),
    ('FLASHLIGHT','LED Flashlight',         'Emergency'),
    ('CANDLES',   'Emergency Candles',      'Emergency'),
    ('TARP',      'Heavy Duty Tarp',        'Emergency'),
    ('BEER-24',   'Light Beer 24-Pack',     'Alcohol'),
    ('MONOPOLY',  'Monopoly Board Game',    'Entertainment'),
    ('CARDS',     'Playing Cards',          'Entertainment'),
    ('STREAMING', 'Roku Streaming Stick',   'Electronics'),
    ('CHARGER',   'Portable Phone Charger', 'Electronics'),
    ('SOUP',      'Canned Soup',            'Grocery'),
    ('BREAD',     'White Bread',            'Grocery'),
    ('PB-16',     'Peanut Butter 16oz',     'Grocery'),
    ('FIRSTAID',  'First Aid Kit',          'Health'),
    ('CHIPS-VP',  'Chips Variety Pack',     'Snacks'),
    ('PTOWEL-6',  'Paper Towels 6-Roll',    'Household'),
    ('TP-12',     'Toilet Paper 12-Roll',   'Household'),
    ('DIAPERS-3', 'Diapers Size 3',         'Baby'),
    ('FORMULA',   'Baby Formula',           'Baby'),
], columns=['product_id', 'product', 'category_clean'])

# Messy category text -> the same 10 categories the database uses
CATEGORY_MAP = {
    'emergency': 'Emergency', 'emergency supplies': 'Emergency',
    'grocery': 'Grocery', 'groceries': 'Grocery',
    'electronics': 'Electronics',
    'entertainment': 'Entertainment', 'games & entertainment': 'Entertainment',
    'household': 'Household', 'household essentials': 'Household',
    'baby': 'Baby', 'baby products': 'Baby',
    'breakfast': 'Breakfast', 'breakfast foods': 'Breakfast',
    'alcohol': 'Alcohol', 'alcoholic beverages': 'Alcohol',
    'snacks': 'Snacks',
    'health': 'Health', 'health & wellness': 'Health',
}
def norm_category(s):
    return s.astype(str).str.strip().str.lower().map(CATEGORY_MAP)


# ===== TRANSACTIONS =====
txn = raw_txn.copy()
n0 = len(txn)

dups = txn.duplicated().sum()
txn = txn.drop_duplicates().reset_index(drop=True)
log('txn_duplicates', 'Dropped exact duplicate rows (dup transaction_ids were all exact copies)', dups)
assert txn.transaction_id.is_unique

before = txn.store_id.copy()
txn['store_id'] = norm_store_id(txn.store_id)
log('txn_store_id', "Normalized store_id to 'FL-###' (fixed case, spaces, '_' and missing dash)",
    (before != txn.store_id).sum())
assert txn.store_id.str.fullmatch(r'FL-\d{3}').all() and txn.store_id.nunique() == 12

ts = to_na(txn.timestamp)
n_missing_ts = ts.isna().sum()
FORMATS = ['%Y-%m-%d %H:%M:%S', '%Y-%m-%dT%H:%M:%S', '%m/%d/%Y %H:%M',
           '%m-%d-%Y %H:%M:%S', '%d-%b-%Y %H:%M']
parsed = pd.Series(pd.NaT, index=ts.index, dtype='datetime64[ns]')
for fmt in FORMATS:
    parsed = parsed.fillna(pd.to_datetime(ts, format=fmt, errors='coerce'))
unparsed = (ts.notna() & parsed.isna()).sum()
assert unparsed == 0, f'{unparsed} timestamps matched no known format'
txn['timestamp'] = parsed
log('txn_timestamp', 'Parsed 5 date formats (ISO, ISO-T, MM/DD/YYYY, MM-DD-YYYY, DD-Mon-YYYY) to datetime', len(txn) - n_missing_ts)
log('txn_timestamp_missing', 'Missing timestamps (empty/NULL/N/A/NaN) set to NaT; rows KEPT, flagged ts_missing', n_missing_ts)
txn['ts_missing'] = txn.timestamp.isna()
assert txn.timestamp.dropna().between('2024-09-01', '2024-10-01').all()

# Pensacola (FL-710) is Central time; every other store is Eastern. Join weather on timestamp_utc.
CENTRAL = {'FL-710'}
txn['tz'] = np.where(txn.store_id.isin(CENTRAL), 'America/Chicago', 'America/New_York')
txn['timestamp_utc'] = pd.concat([
    g.timestamp.dt.tz_localize(tz, ambiguous='NaT', nonexistent='NaT').dt.tz_convert('UTC')
    for tz, g in txn.groupby('tz')
]).sort_index()
log('txn_timezone', 'Localized timestamps (FL-710 Central, rest Eastern), added timestamp_utc for the weather join',
    txn.timestamp_utc.notna().sum())
txn['date'] = txn.timestamp.dt.normalize()
txn['hour'] = txn.timestamp.dt.hour
txn['dow']  = txn.timestamp.dt.day_name()

                  name
0  daily_sales_summary
1     inventory_levels
=== walmart_transactions ===
25,562 rows x 8 columns

                 dtype  nulls  null_pct  distinct
transaction_id  object      0       0.0     25532
store_id        object      0       0.0        60
timestamp       object    886       3.5     23952
sku             object      0       0.0        45
product_name    object      0       0.0        63
category        object      0       0.0        37
quantity         int64      0       0.0        11
unit_price      object      0       0.0      3668

exact duplicate rows: 30
=== store_locations ===
14 rows x 8 columns

              dtype  nulls  null_pct  distinct
store_id     object      0       0.0        14
store_name   object      0       0.0        12
city         object      0       0.0        12
state        object      0       0.0         3
zip_code      int64      0       0.0        12
latitude    float64      1       7.1        11
longitude   float64      1 

### 2 — Your decision log

**TODO:** every cleaning decision, what it did, and how many rows it touched. If your teammates disagree with one of these, note the disagreement — that is useful information, not a problem to hide.

In [10]:
# TODO: print or display your decision log
pd.set_option('display.max_colwidth', None)
show_log()

,step,decision,rows
0,txn_duplicates,Dropped exact duplicate rows (dup transaction_ids were all exact copies),30
1,txn_store_id,"Normalized store_id to 'FL-###' (fixed case, spaces, '_' and missing dash)",4345
2,txn_timestamp,"Parsed 5 date formats (ISO, ISO-T, MM/DD/YYYY, MM-DD-YYYY, DD-Mon-YYYY) to datetime",24648
3,txn_timestamp_missing,"Missing timestamps (empty/NULL/N/A/NaN) set to NaT; rows KEPT, flagged ts_missing",884
4,txn_timezone,"Localized timestamps (FL-710 Central, rest Eastern), added timestamp_utc for the weather join",24648


### 3 — SKU consolidation progress

**TODO:** how many distinct SKUs did you start with, how many canonical products do you have now, and what is your largest consolidated group? Show the mapping for at least one product family. Pop-Tarts is a good one to start with.

You do not need this finished — you need it started and defensible.

In [ ]:
# TODO

### 4 — One working weather API call

**TODO:** show the call and the parsed DataFrame. It must include the retry logic and a `source` column. Say which timezone the weather timestamps are in and how you will line them up with the sales timestamps. If it is not working, paste the actual error and what you have tried.

In [ ]:
# Connect to open-meteo
import requests
import pandas as pd, numpy as np
import time

BASE = 'https://archive-api.open-meteo.com/v1/archive'
RETRY_STATUSES = {429, 500, 502, 503, 504}

def daily_weather(start, end, lat=28.54, lon=-81.38, tries=3):
    params = {
        'latitude': lat, 'longitude': lon,
        'start_date': start, 'end_date': end,
        'daily': 'precipitation_sum,windspeed_10m_max',
        'timezone': 'America/New_York',
    }
    for attempt in range(1, tries + 1):
        try:
            r = requests.get(BASE, params=params, timeout=10)
        except (requests.Timeout, requests.ConnectionError):
            if attempt == tries:
                raise
            time.sleep(2 ** attempt)
            continue
        if r.status_code in RETRY_STATUSES:
            if attempt == tries:
                r.raise_for_status()
            time.sleep(2 ** attempt)
            continue
        r.raise_for_status()                       # any other bad status fails loudly
        frame = pd.DataFrame(r.json()['daily'])
        frame = frame.rename(columns={'time': 'date'})
        frame['date'] = pd.to_datetime(frame['date'])
        frame['source'] = 'open-meteo'
        return frame

# A saved copy of the same Open-Meteo data (Orlando, Sept 1-30, 2024) for when you have no wifi
# or the API is down. The source column says so.
FALLBACK = pd.DataFrame({
    'date': pd.date_range('2024-09-01', periods=30),
    'precipitation_sum': [3.0, 12.4, 24.9, 22.1, 23.1, 9.3, 6.8, 94.7, 28.0, 14.4,
                          12.0, 0.7, 0.6, 21.1, 1.2, 7.9, 2.3, 13.3, 1.4, 0.7,
                          0.4, 2.2, 1.3, 0.3, 6.4, 15.2, 2.5, 18.0, 4.6, 3.3],
    'windspeed_10m_max': [18.8, 13.5, 22.8, 15.5, 15.2, 9.0, 14.2, 12.6, 11.5, 14.3,
                          15.8, 12.1, 10.4, 17.4, 9.0, 13.3, 12.4, 9.3, 12.9, 15.3,
                          19.9, 17.9, 16.6, 17.7, 27.2, 48.8, 45.1, 19.8, 11.7, 8.6],
    'source': 'fallback sample',
})

try:
    weather_live = daily_weather('2024-09-01', '2024-09-30')
except requests.RequestException as err:
    print('API call failed:', err)
    weather_live = FALLBACK

same = (np.allclose(weather_live['precipitation_sum'], FALLBACK['precipitation_sum'])
        and np.allclose(weather_live['windspeed_10m_max'], FALLBACK['windspeed_10m_max']))
print('source:', weather_live['source'].iloc[0])
print('matches the saved copy:', same)
weather_live.head()



The timezone is America/New York. I will line them up with the sales timestamp by adding a date column to the sales table. I will do a left join on these two tables, then.

### 5 — One chart

**TODO:** daily order volume across your date range. It does not need to be presentation quality yet, but it does need labeled axes and a title.

Look at it before you submit. Does the shape make sense? An early chart is the fastest way to catch a cleaning bug — if your daily volume has a day at zero or a day at ten times everything else, that is usually your pipeline, not the hurricane.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Load the raw data
tx = pd.read_csv(REPO_PATH + '/data/walmart_transactions.csv')

# Read the timestamps (they come in 3 formats, 'mixed' handles all of them)
tx['ts'] = pd.to_datetime(tx['timestamp'], format='mixed', errors='coerce')

# Drop rows with no timestamp
tx = tx.dropna(subset=['ts'])

# Keep just the day, then count unique orders per day
tx['date'] = tx['ts'].dt.date
daily = tx.groupby('date')['transaction_id'].nunique()

# Plot
plt.figure(figsize=(10, 4.5))
plt.plot(daily.index, daily.values, marker='o')
plt.axvline(pd.Timestamp('2024-09-26'), color='orange', linestyle='--', label='Landfall (Sept 26)')
plt.xlabel('Date')
plt.ylabel('Orders per day')
plt.title('Daily order volume, Sept 2024 (pre-cleaning)')
plt.legend()
plt.xticks(rotation=45)
plt.show()

**Note:**

This is a first look at the shape of daily order volume, made on the raw data before cleaning. We only parsed the mixed timestamp formats, dropped 886 rows with no timestamp, and counted unique transaction_ids per day. The shape already matches the Helene timeline (flat baseline, peak on Sept 24–25, drop at landfall), with no zero days or extreme outliers. We will rerun this chart on the cleaned data once the cleaning pipeline is finished.

### 6 — Self-check

Fill in what you have and leave the rest. **The required items are the team setup and the charter.** Everything else is progress we want to see, not a pass or fail. The checks are shape checks; they cannot verify your analysis is right, only that you have gotten far enough to have one.

In [ ]:
required = {
    'team_setup_complete': check_team(team_setup, quiet=True),   # from the team setup section
    'charter_committed': True,         # TODO: True once CHARTER.md shows in the git log above
}

progress = {
    'rows_after_cleaning': None,        # an integer
    'decisions_logged': None,           # how many entries in your log
    'canonical_products': None,         # distinct products after consolidation
    'weather_api_rows': None,           # rows returned by your API call
    'notebook_runs_clean': False,       # True after Restart and Run All
}

def is_done(value):
    return not (value is None or value is False
                or (isinstance(value, str) and value.startswith('TODO')))

def check(required, progress):
    print('REQUIRED')
    for label, value in required.items():
        print('  PASS' if is_done(value) else '  FAIL', '-', label, '=', value)
    print('PROGRESS (fill in what you have)')
    for label, value in progress.items():
        print('  done' if is_done(value) else '  not yet', '-', label, '=', value)
    ok = all(is_done(v) for v in required.values())
    print()
    print('required items complete:', ok)
    print('progress items done:', sum(is_done(v) for v in progress.values()), 'of', len(progress))
    return ok

check(required, progress)

### 7 — Risk and plan

Two questions, answered concretely.

**a)** What is the single most likely thing to go wrong in the next two weeks? Name a specific technical or team problem, not "running out of time."

Timing in work schedule will lead to stresses. Dependencies of others will not be completed until the person that is dependent on it is unable to work on the project. We have combatted this potential by sharing when we are unable to work next week (e.g., two members will be away for the entire weekend).

**b)** What are you doing about it this week? Name who is doing it and by when.

As mentioned above, we are sharing when we are available so that dependencies will be finished in reasonable time for our partners.

**Sophie:** I will be writing the code for chart portion; restart and run the session once everything is filled out; and commit changes to our repo.

### 8 — Where each question stands

**TODO:** mark each of the five questions. These owners must match the question owners table in your charter. Be honest — this is the section I read first.

| Question | Status | Who owns it |
|---|---|---|
| Q1 demand surge | not started / in progress / draft answer | _..._ |
| Q2 SKU consolidation | _..._ | _..._ |
| Q3 weather correlation | Blocked / Waiting on consolidation | AJ Amos |
| Q4 store and geography | Not Started | _..._ |
| Q5 signal or artifact | Not Started | Sophie Li |

At the end of week 1, "not started" on Q4 and Q5 is expected. "Not started" on all five means come find me on Monday.